# EDA y Limpieza de Datos

**TFM:** Sistema de Apoyo a la Decisión Clínica en Oncología Pediátrica  
**Autor:** Alonso Castañón González  
**Dataset:** SEER Research Data — Osteosarcoma y Sarcoma de Ewing pediátrico (2000–2023)

**Objetivo de este notebook:** Realizar la carga, inspección inicial, limpieza y preparación del dataset de SEER
para su uso en el modelo predictivo de supervivencia.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='muted')

SEED = 2026
np.random.seed(SEED)

print("Entorno configurado correctamente")

Entorno configurado correctamente


## 1) Carga de datos

In [2]:
# Ruta del dataset del SEER
DATA_PATH = '../data/seer_osteosarcoma_ewing.csv'

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
df.head(5)

Shape: (4436, 14)
Filas: 4,436
Columnas: 14


,Age recode with <1 year olds and 90+,Sex,Year of diagnosis,Histologic Type ICD-O-3,Primary Site,Combined Summary Stage with Expanded Regional Codes (2004+),CS tumor size (2004-2015),RX Summ--Surg Prim Site (1998-2022),Radiation recode,"Chemotherapy recode (yes, no/unk)",Survival months,Survival months flag,SEER cause-specific death classification,Vital status recode (study cutoff used)
0,05-09 years,Female,2004,9260,400,Regional by direct extension only,064,25,Beam radiation,Yes,239,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
1,15-19 years,Male,2001,9260,413,Blank(s),Blank(s),30,None/Unknown,Yes,275,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
2,10-14 years,Female,2001,9260,414,Blank(s),Blank(s),00,Beam radiation,Yes,119,Complete dates are available and there are mor...,Dead (attributable to this cancer dx),Dead
3,10-14 years,Female,2001,9180,402,Blank(s),Blank(s),30,None/Unknown,Yes,274,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
4,15-19 years,Female,2000,9181,414,Blank(s),Blank(s),00,Beam radiation,Yes,23,Complete dates are available and there are mor...,Dead (attributable to this cancer dx),Dead


## 2) Inspección inicial del dataset

In [3]:
# Tipos de datos y valores no nulos por columna
print("---- TIPOS DE DATOS ----")
print(df.dtypes)
print(f"\n---- VALORES NO NULOS ----")
print(df.notnull().sum())

---- TIPOS DE DATOS ----
Age recode with <1 year olds and 90+                           object
Sex                                                            object
Year of diagnosis                                               int64
Histologic Type ICD-O-3                                         int64
Primary Site                                                    int64
Combined Summary Stage with Expanded Regional Codes (2004+)    object
CS tumor size (2004-2015)                                      object
RX Summ--Surg Prim Site (1998-2022)                            object
Radiation recode                                               object
Chemotherapy recode (yes, no/unk)                              object
Survival months                                                 int64
Survival months flag                                           object
SEER cause-specific death classification                       object
Vital status recode (study cutoff used)                        ob

## 3) Renombrado de columnas

In [4]:
# Renombrado de columnas para facilitar el trabajo y la identificación
column_mapping = {
    'Age recode with <1 year olds and 90+': 'age_group',
    'Sex': 'sex',
    'Year of diagnosis': 'year_diagnosis',
    'Histologic Type ICD-O-3': 'histology_code',
    'Primary Site': 'primary_site',
    'Combined Summary Stage with Expanded Regional Codes (2004+)': 'stage',
    'CS tumor size (2004-2015)': 'tumor_size_cs',
    'RX Summ--Surg Prim Site (1998-2022)': 'surgery_code',
    'Radiation recode': 'radiation',
    'Chemotherapy recode (yes, no/unk)': 'chemotherapy',
    'Survival months': 'survival_months',
    'Survival months flag': 'survival_months_flag',
    'SEER cause-specific death classification': 'cause_specific_death',
    'Vital status recode (study cutoff used)': 'vital_status'
}

df = df.rename(columns=column_mapping)

print("Columnas renombradas:")
print(df.columns.tolist())

df.head(5)

Columnas renombradas:
['age_group', 'sex', 'year_diagnosis', 'histology_code', 'primary_site', 'stage', 'tumor_size_cs', 'surgery_code', 'radiation', 'chemotherapy', 'survival_months', 'survival_months_flag', 'cause_specific_death', 'vital_status']


,age_group,sex,year_diagnosis,histology_code,primary_site,stage,tumor_size_cs,surgery_code,radiation,chemotherapy,survival_months,survival_months_flag,cause_specific_death,vital_status
0,05-09 years,Female,2004,9260,400,Regional by direct extension only,064,25,Beam radiation,Yes,239,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
1,15-19 years,Male,2001,9260,413,Blank(s),Blank(s),30,None/Unknown,Yes,275,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
2,10-14 years,Female,2001,9260,414,Blank(s),Blank(s),00,Beam radiation,Yes,119,Complete dates are available and there are mor...,Dead (attributable to this cancer dx),Dead
3,10-14 years,Female,2001,9180,402,Blank(s),Blank(s),30,None/Unknown,Yes,274,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
4,15-19 years,Female,2000,9181,414,Blank(s),Blank(s),00,Beam radiation,Yes,23,Complete dates are available and there are mor...,Dead (attributable to this cancer dx),Dead


## 4) Detección real de valores missing (Blank(s)) y reemplazo por NaN

In [5]:
# Se comprueba la posible existencia de valores perdidos.

missing = df.isnull().sum()
print("---- VALORES MISSING POR COLUMNA INICIALES ----")
print(missing)

---- VALORES MISSING POR COLUMNA INICIALES ----
age_group               0
sex                     0
year_diagnosis          0
histology_code          0
primary_site            0
stage                   0
tumor_size_cs           0
surgery_code            0
radiation               0
chemotherapy            0
survival_months         0
survival_months_flag    0
cause_specific_death    0
vital_status            0
dtype: int64


SEER codifica los missing como el string `Blank(s)`, no como NaN. Necesitamos detectarlos antes de cualquier transformación.

Los valores `None/Unknown` se mantienen como categoría explícita ya que representan casos donde el tratamiento existía pero no fue registrado, lo cual puede ser información clínicamente relevante y distinta de un dato ausente.

In [6]:
# Se identifican los valores desconocidos codificados como Blank(s)

BLANK = 'Blank(s)'

print("---- VALORES BLANK(S) POR COLUMNA ----")
for col in df.columns:
    n_blanks = (df[col].astype(str).str.strip() == BLANK).sum()
    pct = n_blanks / len(df) * 100
    print(f"{col}: {n_blanks} ({pct:.1f}%)")

---- VALORES BLANK(S) POR COLUMNA ----
age_group: 0 (0.0%)
sex: 0 (0.0%)
year_diagnosis: 0 (0.0%)


histology_code: 0 (0.0%)
primary_site: 0 (0.0%)
stage: 727 (16.4%)
tumor_size_cs: 2115 (47.7%)
surgery_code: 122 (2.8%)
radiation: 0 (0.0%)
chemotherapy: 0 (0.0%)
survival_months: 0 (0.0%)
survival_months_flag: 0 (0.0%)
cause_specific_death: 0 (0.0%)
vital_status: 0 (0.0%)


### Conclusiones sobre valores missing

- **CS tumor size (2004-2015)**: 47.7% de missing. Es la variable más problemática.
  Solo tiene datos entre 2004-2015 por diseño del registro SEER. Se conserva en el
  dataset pero **no se usará como predictor en el modelo ML** dado su alto porcentaje
  de missing y su cobertura temporal parcial. Se pueden hacer pruebas con solo la parte
  del dataset que tiene estos valores para ver como afecta su inclusión en las predicciones.

- **Combined Summary Stage (2004+)**: 16.4% de missing. El missing se concentra
  en los años 2000-2003, anteriores al sistema de estadificación combinada.
  Se usará en el modelo imputando o excluyendo estos casos según el análisis posterior.

- **RX Summ--Surg Prim Site (1998-2022)**: 2.8% de missing. Porcentaje bajo y
  manejable. Se imputará o se excluirán estas filas en el pipeline de modelado.

- El resto de columnas están completas al 100%.

In [7]:
# Se reemplaza el string "Blank(s)" por NaN real de pandas.

df = df.replace('Blank(s)', np.nan)

# Se verifica si ahora detecta bien los valores perdidos.
print("---- MISSING VALUES REALES (NaN) POR COLUMNA ----")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({
    'missing': missing,
    'porcentaje': missing_pct
}).query('missing > 0')

print(missing_df)
print(f"\nTotal filas del dataset: {len(df)}")
print(f"Filas completas sin ningún NaN: {df.dropna().shape[0]}")

---- MISSING VALUES REALES (NaN) POR COLUMNA ----
               missing  porcentaje
stage              727        16.4
tumor_size_cs     2115        47.7
surgery_code       122         2.8

Total filas del dataset: 4436
Filas completas sin ningún NaN: 2321


Es importante señalar que los valores perdidos en *stage* y *tumor_size_cs* no siguen un patrón aleatorio (MCAR, Missing Completely At Random), sino que están sistemáticamente asociados al año de diagnóstico del paciente, lo que se corresponde con un mecanismo MAR (Missing At Random) condicionado a esta variable temporal. En el caso de *tumor_size_cs*, la ausencia de valores se concentra en el periodo 2004-2015 debido a cambios en los protocolos de codificación del registro SEER; de forma similar, los nulos en *stage* se concentran en 2000-2003, años previos a la adopción del sistema de estadificación combinada (Combined Summary Stage). Este origen estructural del missingness, ligado a la evolución metodológica del propio registro y no a características clínicas de los pacientes, justifica el tratamiento diferenciado por variable adoptado en este trabajo (exclusión como predictor primario, imputación condicionada o análisis por subconjuntos temporales) frente a una eliminación indiscriminada de filas (dropna global), que supondría descartar cerca de la mitad de la muestra disponible.

## 5) Clasificación de columnas, conversión de tipos y mapeo de códigos

### 5.1) Inspección de valores reales por columna

In [14]:
print("---- VALORES ÚNICOS POR COLUMNA ----")
for col in df.columns:
    n_unique = df[col].nunique()
    sample = df[col].dropna().unique()
    print(f"\n{col} (dtype: {df[col].dtype}) — {n_unique} valores únicos")
    print(f"  Valores:\n {sample}")

---- VALORES ÚNICOS POR COLUMNA ----

age_group (dtype: object) — 5 valores únicos
  Valores:
 ['05-09 years' '15-19 years' '10-14 years' '00 years' '01-04 years']

sex (dtype: object) — 2 valores únicos
  Valores:
 ['Female' 'Male']

year_diagnosis (dtype: int64) — 24 valores únicos
  Valores:
 [2004 2001 2000 2002 2003 2005 2006 2007 2012 2008 2009 2010 2011 2022
 2013 2014 2015 2020 2016 2017 2018 2019 2021 2023]

histology_code (dtype: int64) — 9 valores únicos
  Valores:
 [9260 9180 9181 9186 9182 9183 9187 9185 9184]

primary_site (dtype: int64) — 60 valores únicos
  Valores:
 [400 413 414 402 412 493 403 410 480 495 491 492 409 714 419 490 496  19
 720 649 749 761 401 494 519 418 715 548 411 809 311 382 719 499 712 445
 529 696 408  79 721 220 384 501 343 711 109 310 475 380 341 569 471 621
 701 729 381 713 509 619]

stage (dtype: object) — 6 valores únicos
  Valores:
 ['Regional by direct extension only' 'Distant site(s)/node(s) involved'
 'Localized only'
 'Regional by both di

### 5.2) Conclusiones de la inspección y plan de acción

Tras observar los valores iniciales, se toman las siguientes decisiones:

| Columna | Tipo actual | Tipo real | Acción |
|---|---|---|---|
| `age_group` | object | Categórica ordinal | Convertir a Categorical con orden (puede ser útil para visualizaciones y algunos modelos)|
| `sex` | object | Categórica nominal | Convertir a Categorical |
| `year_diagnosis` | int64 | Numérica discreta | Mantener como int64 |
| `histology_code` | int64 | Categórica nominal (código) | Mapear a nombre de tumor (publicados en el diccionario ICD-O-3 por la OMS y SEER) |
| `primary_site` | int64 | Categórica nominal (código) | Mapear a localización anatómica simplificada (códigos C40-C41 del ICD-O-3 o agrupación por categorías clínicas como huesos largos, huesos planos, pelvis, etc. ???) |
| `stage` | object | Categórica ordinal | Limpiar categorías y convertir a Categorical con orden |
| `tumor_size_cs` | object | Numérica continua (mm) | Convertir a float, tratar códigos especiales (999, 997, 000...) |
| `surgery_code` | object | Categórica nominal (código) | Mapear a descripción de cirugía (publicadas en seer.cancer.gov/tools/surgery) |
| `radiation` | object | Categórica nominal | Simplificar categorías |
| `chemotherapy` | object | Categórica nominal | Mantener Yes/No/Unknown |
| `survival_months` | int64 | Numérica continua | Mantener como int64 |
| `survival_months_flag` | object | Categórica nominal | Simplificar a código corto |
| `cause_specific_death` | object | Categórica nominal | Simplificar categorías |
| `vital_status` | object | Categórica nominal | Convertir a Categorical |

### 5.3) Conversión de tipos y transformaciones por columna

#### age_group

In [15]:
# Categórica ordinal con orden natural de edad
age_order = ['00 years', '01-04 years', '05-09 years', '10-14 years', '15-19 years']

df['age_group'] = pd.Categorical(df['age_group'], categories=age_order, ordered=True)

print(df['age_group'].dtype)
print(df['age_group'].cat.categories)
print(df['age_group'].value_counts().sort_index())

category
Index(['00 years', '01-04 years', '05-09 years', '10-14 years', '15-19 years'], dtype='object')
age_group
00 years         26
01-04 years     176
05-09 years     757
10-14 years    1746
15-19 years    1731
Name: count, dtype: int64


#### sex

In [ ]:
# Categórica nominal con dos valores
df['sex'] = pd.Categorical(df['sex'])

print(df['sex'].dtype)
print(df['sex'].value_counts())

category
sex
Male      2558
Female    1878
Name: count, dtype: int64


#### year_diagnosis

In [18]:
# Numérica discreta, se verifica el rango

print(f"Rango: {df['year_diagnosis'].min()} — {df['year_diagnosis'].max()}")
print(f"Dtype: {df['year_diagnosis'].dtype}")

Rango: 2000 — 2023
Dtype: int64


#### histology_code

Fuente: National Cancer Institute, Surveillance, Epidemiology, and End Results Program. ICD-O-3 SEER Site/Histology Validation List. April 29, 2022.
Disponible en: seer.cancer.gov/icd-o-3/

In [19]:
# Código ICD-O-3 numérico que se mapea a nombre de tumor mediante el diccionario ICD-O-3 OMS/SEER

histology_map = {
    9180: 'Osteosarcoma NOS',
    9181: 'Chondroblastic osteosarcoma',
    9182: 'Fibroblastic osteosarcoma',
    9183: 'Telangiectatic osteosarcoma',
    9184: 'Osteosarcoma in Paget disease',
    9185: 'Small cell osteosarcoma',
    9186: 'Central osteosarcoma',
    9187: 'Intraosseous well differentiated osteosarcoma',
    9260: 'Ewing sarcoma'
}

df['tumor_type'] = df['histology_code'].map(histology_map)
df['tumor_type'] = pd.Categorical(df['tumor_type'])

print(df['tumor_type'].value_counts())
print(f"\nCódigos sin mapear: {df['tumor_type'].isnull().sum()}")

tumor_type
Osteosarcoma NOS                                 2058
Ewing sarcoma                                    1649
Chondroblastic osteosarcoma                       410
Telangiectatic osteosarcoma                       108
Central osteosarcoma                              103
Fibroblastic osteosarcoma                          76
Small cell osteosarcoma                            25
Intraosseous well differentiated osteosarcoma       6
Osteosarcoma in Paget disease                       1
Name: count, dtype: int64

Códigos sin mapear: 0


Los subtipos minoritarios de osteosarcoma (telangiectásico, condroblástico, fibroblástico, etc.) presentan un número de casos muy reducido en comparación con "Osteosarcoma NOS". Se mantienen como categorías independientes en esta fase. En el notebook de modelado se evaluará el impacto de agruparlos bajo una única categoría "Osteosarcoma" comparando el rendimiento del modelo en ambos escenarios.